In [11]:
pip install gitsource toyaikit minsearch openai python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
import os
from openai import OpenAI

In [14]:
from gitsource import GithubRepositoryDataReader, chunk_documents
import minsearch

In [15]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools

In [16]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [17]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for f in files:
    documents.append(f.parse())

print("Q1 pages:", len(documents))

Q1 pages: 72


In [18]:
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

results = index.search(
    "How does the agentic loop keep calling the model until it stops?"
)

print("Q2 top file:", results[0]["filename"]) 

Q2 top file: 01-agentic-rag/lessons/14-agentic-loop.md


In [19]:
def build_context(docs):
    return "\n\n".join(
        f"filename: {d['filename']}\ncontent: {d['content']}"
        for d in docs[:3]
    )


def rag(query):
    results = index.search(query)[:3]

    context = build_context(results)

    prompt = f"""
Answer using context:

{context}

Question: {query}
"""

    resp = openai_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return resp.choices[0].message.content, resp.usage.prompt_tokens


answer1, tokens1 = rag("How does the agentic loop work?")
print("Q3 tokens:", tokens1) 

Q3 tokens: 4180


In [20]:
chunks = chunk_documents(documents, size=2000, step=1000)
print("Q4 chunks:", len(chunks))


chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)


def rag_chunks(query):
    results = chunk_index.search(query)[:3]

    context = "\n\n".join(
        f"filename: {r['filename']}\ncontent: {r['content'][:1500]}"
        for r in results
    )

    prompt = f"""
Answer using context:

{context}

Question: {query}
"""

    resp = openai_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return resp.choices[0].message.content, resp.usage.prompt_tokens


answer2, tokens2 = rag_chunks("How does the agentic loop work?")
print("Q5 reduction:", tokens1 - tokens2)

Q4 chunks: 295
Q5 reduction: 3026


In [ ]:
search_count = 0

def search(query: str) -> list:
    global search_count
    search_count += 1
    return chunk_index.search(query)


tools = Tools()
tools.add_tool(search)

client = OpenAIClient(
    client=openai_client,
    model="llama-3.3-70b-versatile"
)

messages = [
    {"role": "system", "content": """
You are an agentic assistant.

You MUST use search tool multiple times before answering.
"""},

    {"role": "user", "content": "How does the agentic loop work and how is it different from plain RAG?"}
]

response = client.send_request(
    chat_messages=messages,
    tools=tools
)

print("RAW RESPONSE:", response)

tool_calls = response.output

for item in tool_calls:
    if item.type == "function_call" and item.name == "search":
        args = eval(item.arguments)
        search(args["query"])

print("Q6 tool calls (real python execution):", search_count)

RAW RESPONSE: Response(id='resp_01kv679nb6e5bam0k4qdqkykm5', created_at=1781546669.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='llama-3.3-70b-versatile', object='response', output=[ResponseReasoningItem(id='resp_01kv679nb6e5bvk4edx780c8hh', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop vs plain RAG"}', call_id='nq9rgdr5p', name='search', type='function_call', id='nq9rgdr5p', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop definition"}', call_id='ycdmacxrd', name='search', type='function_call', id='ycdmacxrd', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"plain RAG definition"}', call_id='j8e23dy3b', name='search', type='function_call', id='j8e23dy3b', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionToo